In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [ ]:
from pyspark.sql import functions as F
import csv
import os
import pandas as pd
from datetime import datetime

order_items_path = "order_items_final_20260331_224715.csv"
products_path = "products_final_20260331_224555.csv"

order_items_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(order_items_path)
    .dropDuplicates()
    .dropna(subset=["product_id"])
    .withColumn("product_id", F.trim(F.col("product_id")))
)

products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(products_path)
    .dropDuplicates()
    .dropna(subset=["product_id"])
    .withColumn("product_id", F.trim(F.col("product_id")))
)

joined_df = order_items_df.join(products_df, on="product_id", how="inner")

joined_df.printSchema()
print("rows:", joined_df.count())

joined_df = joined_df.drop("product_id")
joined_pd = joined_df.toPandas()
output_dir = os.getcwd()

output_path = os.path.join(
    output_dir, f"order_items_products_final_{datetime.now():%Y%m%d_%H%M%S}.csv"
 )
joined_pd.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

root
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- category_name: string (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- volume_cm3: double (nullable = true)

rows: 100516
CSV salvo em: c:\Users\Sofhia\Downloads\archive\order_items_products_final_20260331_230122.csv
